# Decision Tree & XGBoost  - Modelos Globales (PreTec21 → Tec21)
## Predicción de Deserción Estudiantil · Equipo 7

---
**Prerrequisito:** `clustering/02_kmeans_independiente.ipynb`

Entrena **Decision Tree** y **XGBoost** con búsqueda de hiperparámetros para XGB,
validación cruzada 5-fold y evaluación cross-regime en Tec21.
Cada modelo muestra su matriz de confusión, curvas ROC/PR y métricas completas.


## Setup e Importaciones

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pickle
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection  import (StratifiedKFold, cross_val_score,
                                       RandomizedSearchCV, train_test_split)
from sklearn.metrics          import (roc_auc_score, recall_score, f1_score,
                                       precision_score, roc_curve, auc,
                                       precision_recall_curve, average_precision_score,
                                       ConfusionMatrixDisplay, confusion_matrix)
from sklearn.impute           import SimpleImputer
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(' XGBoost:', xgb.__version__)
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost no disponible')

SEED = 42
np.random.seed(SEED)
print(' Librerías cargadas')

 XGBoost: 3.2.0
 Librerías cargadas


## Configuración y carga de datos

In [3]:
PROCESSED_DIR = Path('../../../data/processed')
IMG_DIR       = Path('images'); IMG_DIR.mkdir(exist_ok=True)

TARGET    = 'retention'
MIN_AUC   = 0.60

df_pre = pd.read_csv(PROCESSED_DIR / 'df_pre.csv')
df_tec = pd.read_csv(PROCESSED_DIR / 'df_tec.csv')
print(f'df_pre: {df_pre.shape}  |  df_tec: {df_tec.shape}')

FileNotFoundError: [Errno 2] No such file or directory: '../../../data/processed/df_pre.csv'

##  8  - Features y Split

In [ ]:
FEATURE_COLS_CANDIDATES = [
    'PNA', 'admission_test_norm', 'english.evaluation', 'admission.rubric',
    'general.math.eval', 'online.test', 'FTE', 'apoyo_financiero',
    'has_extracurriculars', 'has_physical', 'has_cultural', 'has_social',
    'first_gen_enc', 'educ_padres_max', 'socioec_enc', 'social_lag_enc',
    'age', 'is_male', 'estuvo_prepa_tec',
    'foreign_Yes: Foreigner', 'foreign_Yes: National',
    'school_enc', 'region_enc',
    'first_gen_present', 'parents_edu_present', 'took_admission_test',
    'has_socioeconomic_data', 'has_social_lag_data', 'has_zone_data',
]
EXCLUDE = {TARGET, 'cluster', 'generation', 'regime', 'educational.model'}
seen = set()
FEATURE_COLS = [c for c in FEATURE_COLS_CANDIDATES
                if c in df_pre.columns and c not in EXCLUDE
                and c not in seen and not seen.add(c)]
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

def make_split(df_regime, feature_cols, target=TARGET, seed=SEED):
    cols = [c for c in feature_cols if c in df_regime.columns]
    X = df_regime[cols].values.astype(float)
    y = df_regime[target].values.astype(int)
    X = SimpleImputer(strategy='median').fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    return X_tr, X_te, y_tr, y_te, cols

X_tr_pre, X_te_pre, y_tr_pre, y_te_pre, feat_pre = make_split(df_pre, FEATURE_COLS)
X_tr_tec, X_te_tec, y_tr_tec, y_te_tec, feat_tec = make_split(df_tec, FEATURE_COLS)
print(f'PreTec21  train={X_tr_pre.shape}  test={X_te_pre.shape}')
print(f'Tec21     train={X_tr_tec.shape}  test={X_te_tec.shape}')

## Función de evaluación

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name, feat_cols,
                   threshold=None, n_bootstrap=200, seed=SEED):
    model.fit(X_tr, y_tr)
    has_proba = hasattr(model, 'predict_proba')
    y_proba   = model.predict_proba(X_te)[:,1] if has_proba else None
    if threshold is None and has_proba:
        skf_tmp = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
        oof = np.zeros(len(X_tr))
        for tr_i, va_i in skf_tmp.split(X_tr, y_tr):
            model.fit(X_tr[tr_i], y_tr[tr_i])
            oof[va_i] = model.predict_proba(X_tr[va_i])[:,1]
        model.fit(X_tr, y_tr)
        oof_dropout = 1.0 - oof
        prec_oof, rec_oof, thr_oof = precision_recall_curve(y_tr, oof_dropout, pos_label=0)
        f1_oof  = 2*prec_oof*rec_oof/(prec_oof+rec_oof+1e-8)
        thr_best = float(thr_oof[np.argmax(f1_oof[:-1])])
        threshold = 1.0 - thr_best
    y_pred = (y_proba >= threshold).astype(int) if (threshold is not None and has_proba)              else model.predict(X_te)
    auc_val  = roc_auc_score(y_te, y_proba) if has_proba else 0.5
    rec  = recall_score(y_te, y_pred, zero_division=0)
    prec = precision_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, pos_label=0, zero_division=0)
    rng  = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(y_te), len(y_te))
        if y_proba is not None and len(np.unique(y_te[idx])) > 1:
            aucs.append(roc_auc_score(y_te[idx], y_proba[idx]))
    ci_lo, ci_hi = np.percentile(aucs,[2.5,97.5]) if aucs else (auc_val, auc_val)
    print(f'  {model_name:<28} AUC={auc_val:.3f} [{ci_lo:.3f},{ci_hi:.3f}]'
          f'  Recall={rec:.3f}  F1={f1:.3f}  thr={threshold:.2f}')
    return dict(model=model_name, auc=auc_val, ci_lo=ci_lo, ci_hi=ci_hi,
                recall=rec, precision=prec, f1=f1, threshold=threshold,
                y_proba=y_proba, y_pred=y_pred, feat_cols=feat_cols)

##  9a  - Búsqueda de Hiperparámetros XGBoost

Espacio XGB: `n_estimators∈{100,200,300}`, `max_depth∈{3,5,7}`,
`learning_rate∈{0.01,0.05,0.1}`, `subsample∈{0.7,0.8,0.9}`,
`colsample_bytree∈{0.7,0.8,0.9}`. Métrica: AUC-ROC, 5-fold.

In [ ]:
if XGB_AVAILABLE:
    spw = float((y_tr_pre==0).sum()) / max((y_tr_pre==1).sum(),1)
    xgb_param_dist = {
        'n_estimators'    : [100, 200, 300],
        'max_depth'       : [3, 5, 7],
        'learning_rate'   : [0.01, 0.05, 0.1],
        'subsample'       : [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
    }
    xgb_base   = xgb.XGBClassifier(scale_pos_weight=spw, eval_metric='logloss',
                                     random_state=SEED, n_jobs=-1, verbosity=0)
    xgb_search = RandomizedSearchCV(
        xgb_base, xgb_param_dist, n_iter=30, scoring='roc_auc',
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        random_state=SEED, n_jobs=-1, verbose=0
    )
    xgb_search.fit(X_tr_pre, y_tr_pre)
    best_xgb_params = xgb_search.best_params_
    print('✓ Mejores parámetros XGB:')
    for k, v in sorted(best_xgb_params.items()):
        print(f'    {k}: {v}')
    print(f'  CV AUC (media): {xgb_search.best_score_:.4f}')
else:
    best_xgb_params = {}
    print('XGBoost no disponible')

##  9b  - Validación Cruzada 5-fold

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

dt_cv = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=SEED)
scores_dt = cross_val_score(dt_cv, X_tr_pre, y_tr_pre, cv=skf, scoring='roc_auc', n_jobs=-1)
print(f'CV AUC 5-fold DecisionTree   (PreTec21): {scores_dt.mean():.4f} ± {scores_dt.std():.4f}')

if XGB_AVAILABLE:
    spw = float((y_tr_pre==0).sum()) / max((y_tr_pre==1).sum(),1)
    xgb_cv = xgb.XGBClassifier(**best_xgb_params, scale_pos_weight=spw,
                                 eval_metric='logloss', random_state=SEED, n_jobs=-1, verbosity=0)
    scores_xgb = cross_val_score(xgb_cv, X_tr_pre, y_tr_pre, cv=skf, scoring='roc_auc', n_jobs=-1)
    print(f'CV AUC 5-fold XGBoost        (PreTec21): {scores_xgb.mean():.4f} ± {scores_xgb.std():.4f}')

##  9c  - Decision Tree: entrenamiento y evaluación

In [ ]:
dt_final = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=SEED)
print('Evaluando Decision Tree (PreTec21 → Tec21):')
res_dt = evaluate_model(dt_final, X_tr_pre, X_te_tec, y_tr_pre, y_te_tec,
                        'DecisionTree', feat_pre)

### Decision Tree  - Matriz de Confusión y Curvas ROC/PR

In [ ]:
#  Matriz de Confusión  
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_te_tec, res_dt['y_pred'], normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Desertor','Retenido'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues', values_format='.2f')
axes[0].set_title(f'Matriz de Confusión (norm.)\nAUC={res_dt["auc"]:.3f}')

 # Curvas ROC y PR  
fpr, tpr, _ = roc_curve(y_te_tec, res_dt['y_proba'])
roc_auc_val  = auc(fpr, tpr)
prec_c, rec_c, _ = precision_recall_curve(y_te_tec, res_dt['y_proba'])
ap_val = average_precision_score(y_te_tec, res_dt['y_proba'])

axes[1].plot(fpr, tpr, color='#4C72B0', lw=2, label=f'ROC (AUC={roc_auc_val:.3f})')
axes[1].plot([0,1],[0,1],'k--', lw=1, alpha=0.5)
ax2 = axes[1].twinx()
ax2.plot(rec_c, prec_c, color='#DD8452', lw=2, linestyle='--', label=f'PR (AP={ap_val:.3f})')
axes[1].set_xlabel('FPR / Recall'); axes[1].set_ylabel('TPR', color='#4C72B0')
ax2.set_ylabel('Precisión', color='#DD8452')
axes[1].set_title('Curvas ROC y Precision-Recall (Tec21)')
lines1, labs1 = axes[1].get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, labs1+labs2, loc='lower left', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(IMG_DIR / 'dt_metricas.png', dpi=150, bbox_inches='tight')
plt.show()
print(f' Guardado: {IMG_DIR}/dt_metricas.png')

 # Métricas resumen  
print('\n Métricas Finales  - DT ')
print(f'\tAUC-ROC  : {res_dt["auc"]:.4f} [{res_dt["ci_lo"]:.3f}, {res_dt["ci_hi"]:.3f}]')
print(f'\tRecall   : {res_dt["recall"]:.4f}')
print(f'\tPrecision: {res_dt["precision"]:.4f}')
print(f'\tF1 (0)   : {res_dt["f1"]:.4f}')
print(f'\tUmbral   : {res_dt["threshold"]:.4f}')

##  9c  - XGBoost: entrenamiento y evaluación

In [ ]:
if XGB_AVAILABLE:
    spw = float((y_tr_pre==0).sum()) / max((y_tr_pre==1).sum(),1)
    xgb_final = xgb.XGBClassifier(**best_xgb_params, scale_pos_weight=spw,
                                    eval_metric='logloss', random_state=SEED,
                                    n_jobs=-1, verbosity=0)
    print('Evaluando XGBoost (PreTec21 a Tec21):')
    res_xgb = evaluate_model(xgb_final, X_tr_pre, X_te_tec, y_tr_pre, y_te_tec,
                              'XGBoost', feat_pre)
else:
    print('XGBoost no disponible, omitido')
    res_xgb = None

### XGBoost  - Matriz de Confusión y Curvas ROC/PR

In [ ]:
if res_xgb is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    cm = confusion_matrix(y_te_tec, res_xgb['y_pred'], normalize='true')
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Desertor','Retenido'])
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues', values_format='.2f')
    axes[0].set_title(f'XGBoost  - Matriz de Confusión (norm.)\nAUC={res_xgb["auc"]:.3f}')

    fpr, tpr, _ = roc_curve(y_te_tec, res_xgb['y_proba'])
    roc_auc_val  = auc(fpr, tpr)
    prec_c, rec_c, _ = precision_recall_curve(y_te_tec, res_xgb['y_proba'])
    ap_val = average_precision_score(y_te_tec, res_xgb['y_proba'])

    axes[1].plot(fpr, tpr, color='#C44E52', lw=2, label=f'ROC (AUC={roc_auc_val:.3f})')
    axes[1].plot([0,1],[0,1],'k--', lw=1, alpha=0.5)
    ax2 = axes[1].twinx()
    ax2.plot(rec_c, prec_c, color='#9467BD', lw=2, linestyle='--', label=f'PR (AP={ap_val:.3f})')
    axes[1].set_xlabel('FPR / Recall'); axes[1].set_ylabel('TPR', color='#C44E52')
    ax2.set_ylabel('Precisión', color='#9467BD')
    axes[1].set_title('XGBoost  - Curvas ROC y PR (Tec21)')
    lines1, labs1 = axes[1].get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    axes[1].legend(lines1+lines2, labs1+labs2, loc='lower left', fontsize=9)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMG_DIR / 'xgb_metricas.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Guardado: {IMG_DIR}/xgb_metricas.png')

    print('\n Métricas Finales  - XGBOOST ')
    print(f'\tAUC-ROC  : {res_xgb["auc"]:.4f} [{res_xgb["ci_lo"]:.3f}, {res_xgb["ci_hi"]:.3f}]')
    print(f'\tRecall   : {res_xgb["recall"]:.4f}')
    print(f'\tPrecision: {res_xgb["precision"]:.4f}')
    print(f'\tF1 (0)   : {res_xgb["f1"]:.4f}')
    print(f'\tUmbral   : {res_xgb["threshold"]:.4f}')

## Comparación DT vs XGBoost

In [ ]:
models_to_compare = [('Decision Tree', res_dt)]
if res_xgb is not None:
    models_to_compare.append(('XGBoost', res_xgb))

fig, axes = plt.subplots(1, len(models_to_compare), figsize=(6*len(models_to_compare), 4))
if len(models_to_compare) == 1: axes = [axes]
colors = ['#DD8452', '#C44E52']

for ax, (name, res), color in zip(axes, models_to_compare, colors):
    metrics = ['AUC', 'Recall', 'Precision', 'F1(0)']
    values  = [res['auc'], res['recall'], res['precision'], res['f1']]
    bars = ax.bar(metrics, values, color=color, alpha=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1.05); ax.set_title(name); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comparación de Métricas (evaluación en Tec21)', fontsize=12)
plt.tight_layout()
plt.savefig(IMG_DIR / 'dt_xgb_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {IMG_DIR}/dt_xgb_comparacion.png')